# dlgskill

> Read, search, and edit dialogs and notebooks through the llmsurgery `Dialog`/`Message` model

#| export
Use this whenever the question or edit concerns a notebook's content: messages, sources, outputs, prompts and their replies.

## The hierarchy

Notebook work happens at three levels, and picking the right level is most of using this module well:

- Content (this module): what the messages say and how they change. `summary_dlg`, `find_msgs`, `view_dlg`, the message editing operations, and `reply2dlg`/`dlg2reply` for a prompt's reply.
- Representation (`fastcore.nbio`, formerly `execnb.nbio`): which keys exist, whether a file is schema-valid, whether bytes changed. Start with `validate_nb`/`validate_cell`, which name the offending cell; use `read_nb` directly when the question is about the dict itself. For *plain* notebooks (no dialog semantics), nbio's `Notebook`/`NbCell` objects and `cell_*` functions are the content surface, and this module's word choice marks the layer: cells for notebooks, messages for dialogs.
- Raw text: only when the file will not parse at all.

Dropping a level is correct exactly when the question is about the representation rather than the content ("why does Jupyter reject this file?", "did that write change any bytes?"). Treat each drop as a signal, though: needing nbio or raw JSON to answer a content question means a higher-level tool was missing. Re-read this module's docs to check it truly is missing, and then propose adding it rather than repeating the workaround.

#| export
## Core APIs

The function/method two-shapes contract is `fastcore.editskill`'s, learned once: the function is a transaction addressed by `dlg=` (an ipynb path, or None meaning the current dialog file: `set_dlg`); the method is a session on a held `Dialog` or `Message`, saved by an explicit `dlg.save()` (`msg_str_replace(id, ..., dlg=p)` ⟷ `m.str_replace(...)`). Wrapping any message list in an ephemeral `Dialog(msgs)` makes the whole session surface available on it.

- `summary_dlg(dlg)` / `d.summary()`: one preview line per message, `id:t:content` (t: c=code n=note p=prompt r=raw).
- `find_msgs(pattern, dlg, ...)`: search by regex, type, error state, heading, ids, or a `pred` (`symdef_finder`/`symref_finder`/`ast_finder` build structural ones); `context` defaults to 1 (the neighbouring message usually explains the match). Returns `MsgRow` snapshots (`id`, `msg_type`, `content`, `out`, `meta`), shown as preview lines. `d.find_msgs(...)` returns live `Message` objects in a `Msgs` list instead, edited directly rather than re-addressed. Every live message carries a `dlg` backref to its owning `Dialog`, so dialog-level operations are always in reach from a message in hand (e.g. `m.dlg.save()` after mutating `m.output` directly).
- `view_dlg(dlg)` / `d.view()` / `view_msg(id)` / `m.view()` / `view_msgs(*ids)` / `msg2xml(m)` / `m.to_xml()`: full views in the shared `item2xml` grammar (a prompt's reply is its `<out>` section); `incl_out=True` on the line views appends the message's output the same way.
- Structure: `add_msg`, `del_msgs`, `move_msgs`, `split_msg`, `merge_msgs`, `copy_msgs`/`cut_msgs`/`paste_msgs`, `create_dlg`, with session twins `d.move_msgs`, `m.split`, `d.merge_msgs`, `d.copy_msgs`/`d.cut_msgs`/`d.paste_msgs` (session adds go through `d.mk_message`, deletes through `d.remove_msgs`); the `%%add_msg` magic takes its body verbatim: its line is `%%add_msg [dlg] [msg_type] [before=|after=<id>]`, where a bare path token is the dlg and a bare type name the msg_type, and keyword spellings win over bare tokens.
- Text edits: `msg_str_replace`, `msg_strs_replace`, `msg_insert_line`, `msg_replace_lines`, `msg_del_lines`, `msg_ast_replace` (all with `re_filter`/line-range powers; `out=True` edits a prompt's reply or a code message's outputs literal), with the same names as `Message` methods for in-memory editing; `lnhashview_msg`/`msg_exhash` (and `m.lnhashview()`/`m.exhash()`) for hash-verified line edits (`lnhashview_msg` is `view_msg(..., lnhashs=True)`; only the exhash pair needs the `exhash` package).
- `reply2dlg(pmsg)`/`dlg2reply(sub)`: explode a reply into note/code messages and back; byte-exact for fmt2hist-clean replies.

#| export
## Idiomatic usage

Start by registering the notebook: `set_dlg(path)` makes every function here default to it (and clikernel's `%nbrun` magic follows suit), so calls read as "do this to message X". Then orient before acting:

- `summary_dlg()` first for anything sizable: a cheap one-line-per-message map. `view_dlg()` when you need the full story in order with ids. Read a notebook in full before describing or changing what it does - the interleaved prose, examples, and stored outputs (`incl_out=True`) are the design rationale.
- Read before probing: an ad-hoc "what happens if..." experiment usually re-derives, more slowly and less reliably, what an existing example cell already shows. If after reading you still need an experiment, that's a gap in the notebook - add it as a proper example cell so the next reader doesn't repeat it.
- `find_msgs` is the targeted view: keep the default `context=1` (the neighbouring markdown usually explains the match), and `ids=` with context is the positional query ("does A precede B?", "what's the idiom around here?"). Name questions are structural, not textual: `symdef_finder`/`symref_finder`/`ast_finder` beat regexes over binding syntax. Where available, rgapi's `nbrg` searches cell sources across files and returns the cell ids these functions take.
- Edit at the right level: `add_msg`/`%%add_msg` for new messages; within a message, prefer hash-verified addressing (`lnhashview_msg`/`msg_exhash`, or exhash's `lnhashview_cell`/`cell_exhash` by path and cell id) - view with the lnhash variant the moment an edit is plausible, so the view doubles as the edit's address book. The plain `msg_*` editors fit where exhash can't express the edit, e.g. one `msg_str_replace` across an id list. Never splice via `read_nb`/`write_nb` internals: if a primitive is missing here, stop and propose adding it.
- `dlg.validate()` catches model-level damage early; `dlg.save()` writes back. Don't wrap calls in defensive scaffolding (`hasattr`, `try/except`) - call directly and read the bare result.

In [ ]:
#| default_exp dlgskill

In [ ]:
#| export
import shlex, re, copy
from fastcore.utils import *
from fastcore.meta import splice_sig, delegates
from fastcore.xtras import str_diff
from fastcore.xml import to_xml
from fastcore.nbio import item2xml
from fastcore.tools import insert_line, str_replace, strs_replace, replace_lines, del_lines, ast_replace, lnhash
from llmsurgery.dialog import *
from llmsurgery.ipynb import read_ipynb, write_ipynb
from llmsurgery.hist import reply2dlg, dlg2reply, render_outputs_ai  # chkstyle: ignore (re-exported via _all_)
_all_ = ['reply2dlg', 'dlg2reply']

In [ ]:
from fastcore.test import *
from tempfile import mkdtemp

## The current dialog, and finding messages

In [ ]:
#| export
_cur_dlg = None

def set_dlg(
    fname, # ipynb path that later calls will default to
):
    "Set the current dialog file, used by these functions when `dlg` is None"
    global _cur_dlg
    _cur_dlg = Path(fname).resolve()
    return _cur_dlg

def cur_dlg():
    "A fresh `Dialog` read from the current dialog file (None if unset): the session-side entry to the ambient default"
    return read_ipynb(_cur_dlg) if _cur_dlg else None

def _to_dlg(x):
    "The dialog file `x` (None: the current dialog file), read fresh from disk"
    if isinstance(x, Dialog): raise TypeError('dlg= takes an ipynb path; on a live Dialog, call the method instead')
    if x is None: x = _cur_dlg
    if x is None: raise ValueError('No dialog: pass a path, or call set_dlg() first')
    return read_ipynb(x)

def _load_dlg(x):
    "`(dialog, True)`: read `x` from disk; the second element marks it ours to save"
    return _to_dlg(x), True

def _save(dlg, sv=True):
    "Write back to the file this call loaded from"
    if sv: dlg.save()

In [ ]:
#| export
@patch
def msg(self:Dialog,
    id, # A `Message` (matched by its id), or an id: exact, or unique prefix
):
    "The matching `Message` in this dialog"
    if isinstance(id, Message): id = id.id
    ms = [m for m in self.messages if m.id==id]
    if not ms: ms = [m for m in self.messages if m.id.startswith(id)]
    if len(ms) != 1: raise KeyError(f"{'ambiguous' if ms else 'no'} message id: {id!r}")
    return ms[0]

On-disk cell ids are message ids without the leading underscore, so `msg` accepts either form, and unique prefixes work like everywhere else in this ecosystem. The fixture below is written to disk too, since path-based calls re-read the file per call:

In [ ]:
tdir = Path(mkdtemp())
d = Dialog(name='demo')
setup = d.mk_message('# Setup', msg_type=snote)
calc = d.mk_message('x = 21\nx*2', msg_type=scode, output=code_output('42'))
ask = d.mk_message('Double it.', msg_type=sprompt, output='It is **42**.')
bad = d.mk_message('1/0', msg_type=scode, output=[dict(output_type='error', ename='E', evalue='boom', traceback=['tb'])])
write_ipynb(d, tdir/'demo.ipynb')
test_eq(d.msg(calc.id), calc)
test_eq(d.msg(calc.id[:5]), calc)  # unique prefixes work too
with expect_fail(KeyError, 'no message'): d.msg('zzzz')

## Views

In [ ]:
#| export
@patch
def summary(self:Dialog,
    maxlen:int=120, # Maximum characters per line
):
    "One `preview` line per message: `id:t:content` (t: c=code n=note p=prompt r=raw)"
    return self.messages.show(maxlen)

def summary_dlg(
    dlg=None, # An ipynb path; the current dialog file if None
    maxlen:int=120, # Maximum characters per line
):
    "One `preview` line per message of the dialog file"
    return _to_dlg(dlg).summary(maxlen)

In [ ]:
s = str(d.summary())
test_eq(len(s.splitlines()), 4)
assert 'reply(13)' in s and '# Setup' in s

In [ ]:
#| export
_t2tag = dict(note='markdown')

def msg2xml(m, incl_out=False, trunc_out=True, ids=True):
    "One message as concise XML: content bare inside its type tag, with an `<out>` section for a code output or a prompt's reply"
    if m.msg_type==sprompt: o = m.ai_res
    elif incl_out and m.msg_type==scode and m.output:
        o = render_outputs_ai(m.output)
        if trunc_out: o = truncstr(o, 512)
    else: o = ''
    it = item2xml(_t2tag.get(m.msg_type, m.msg_type), m.content, o, id=m.id if ids else None,
                  kind=m.meta.get('rec_kind'), meta=m.meta)
    return to_xml(it, do_escape=False)

@patch
def to_xml(self:Message, incl_out=False, trunc_out=True, ids=True):
    "This message as concise XML (`msg2xml`)"
    return msg2xml(self, incl_out, trunc_out, ids)

@patch
def view(self:Dialog,
    incl_out:bool=False, # Include code outputs?
    only_errors:bool=False, # Show only code messages with error outputs (implies `incl_out`)?
    trunc_out:bool=True, # Truncate each output to ~512 chars?
):
    "This dialog as concise XML"
    ms = [m for m in self.messages if m.has_error] if only_errors else self.messages
    body = ''.join(msg2xml(m, incl_out or only_errors, trunc_out) for m in ms)
    return PrettyString(f'<dialog name="{self.name}">{body}</dialog>')

def view_dlg(
    dlg=None, # An ipynb path; the current dialog file if None
    incl_out:bool=False, # Include code outputs?
    only_errors:bool=False, # Show only code messages with error outputs (implies `incl_out`)?
    trunc_out:bool=True, # Truncate each output to ~512 chars?
):
    "The dialog file as concise XML"
    return _to_dlg(dlg).view(incl_out, only_errors, trunc_out)

In [ ]:
v = str(d.view(incl_out=True))
assert '<prompt id="'+ask.id+'">Double it.<out>It is **42**.</out></prompt>' in v  # a prompt's reply is its out
assert '>x = 21\nx*2<out>42</out></code>' in v
verr = str(d.view(only_errors=True))
assert '1/0' in verr and 'Setup' not in verr

In [ ]:
#| export
@patch
def view(self:Message,
    nums:bool=True, # Show line numbers?
    start_line:int=1, # Starting line to view
    end_line:int=None, # End line (defaults to last line if None; -1 for EOF)
    lnhashs:bool=False, # Show exhash `lineno|hash|` addresses instead of line numbers?
    incl_out:bool=False, # Append the output (a prompt's reply, or code outputs) in an `<out>` block?
    trunc_out:bool=True, # Truncate an included output to ~512 chars?
):
    "This message's content with optional line numbers or exhash addresses"
    lines = self.content.splitlines()
    lines = lines[start_line-1:len(lines) if end_line in (None,-1) else end_line]
    res = '\n'.join((lnhash(i,l)+l if lnhashs else f'{i}: {l}' if nums else l) for i,l in enumerate(lines, start_line))
    if incl_out:
        o = self.ai_res if self.msg_type==sprompt else render_outputs_ai(self.output) if self.msg_type==scode and self.output else ''
        if o: res += f"\n<out>\n{truncstr(o, 512) if trunc_out else o}\n</out>"
    return PrettyString(res)

def view_msg(
    id, # Message id, looked up in `dlg` (unique prefixes allowed)
    dlg=None, # An ipynb path; the current dialog file if None
    nums:bool=True, # Show line numbers?
    start_line:int=1, # Starting line to view
    end_line:int=None, # End line (defaults to last line if None; -1 for EOF)
    lnhashs:bool=False, # Show exhash `lineno|hash|` addresses instead of line numbers?
    incl_out:bool=False, # Append the output (a prompt's reply, or code outputs) in an `<out>` block?
    trunc_out:bool=True, # Truncate an included output to ~512 chars?
):
    "Show a message's content with optional line numbers or exhash addresses"
    return _to_dlg(dlg).msg(id).view(nums, start_line, end_line, lnhashs, incl_out, trunc_out)

def view_msgs(
    *ids, # Message ids
    dlg=None, # An ipynb path; the current dialog file if None
    nums:bool=True, # Show line numbers?
    lnhashs:bool=False, # Show exhash `lineno|hash|` addresses instead of line numbers?
    incl_out:bool=False, # Append each output (a prompt's reply, or code outputs) in an `<out>` block?
    trunc_out:bool=True, # Truncate each included output to ~512 chars?
):
    "Show several messages, each preceded by a `# msg <id>` header"
    d = _to_dlg(dlg)
    return PrettyString('\n'.join(f"# msg {(m := d.msg(i)).id}\n{m.view(nums, lnhashs=lnhashs, incl_out=incl_out, trunc_out=trunc_out)}" for i in ids))

In [ ]:
test_eq(str(calc.view()), '1: x = 21\n2: x*2')
test_eq(str(calc.view(start_line=2)), '2: x*2')
test_eq(str(calc.view(lnhashs=True)), f'{lnhash(1, "x = 21")}x = 21\n{lnhash(2, "x*2")}x*2')
assert str(view_msgs(setup.id, calc.id, dlg=tdir/'demo.ipynb', nums=False)).startswith(f"# msg {setup.id}")
test_eq(str(calc.view(incl_out=True)), '1: x = 21\n2: x*2\n<out>\n42\n</out>')
assert 'It is **42**.' in str(ask.view(nums=False, incl_out=True))
assert '<out>' not in str(calc.view())
assert '<out>' in str(view_msg(calc.id, dlg=tdir/'demo.ipynb', incl_out=True))
test_eq(str(view_msgs(calc.id, ask.id, dlg=tdir/'demo.ipynb', incl_out=True)).count('<out>'), 2)

## Searching

In [ ]:
#| export
def _match_head(m, want):
    "Does `m` head the section `want` names? Exact first line when `want` starts with '#', hash-stripped text otherwise"
    if not m.header_level(): return False
    fl = m.content.split('\n', 1)[0]
    if want.startswith('#'): return fl.rstrip()==want.rstrip()
    return fl.lstrip('#').strip()==want.strip()

@patch
def find_msgs(self:Dialog,
    re_pattern:str='', # Regex over content (a prompt's reply included), DOTALL+MULTILINE; an invalid regex matches literally
    msg_type:str=None, # Optional limit by type ('code', 'note', 'prompt', or 'raw')
    only_err:bool=False, # Only code messages with error outputs?
    only_exp:bool=False, # Only exported messages (nbdev export directive in content or meta)?
    ids='', # Optionally filter by ids (comma-separated str, or list); results are always in dialog order, whatever order the ids are given
    before:int=0, # Also include n messages before each match
    after:int=0, # Also include n messages after each match
    context:int=None, # Messages of context around matches (default 1, or 0 when `headers_only`)
    limit:int=None, # Max matched messages
    use_case:bool=False, # Case-sensitive matching?
    use_regex:bool=True, # Regex matching (else plain substring)?
    headers_only:bool=False, # Only heading notes (an outline view)?
    header_section:str=None, # Return the section starting with this heading, plus its children
    pred:callable=None, # Extra match criterion, e.g. from `symdef_finder`/`symref_finder`/`ast_finder`, or host-specific flags
)->Msgs: # Live messages, so results can be edited directly
    "Find this dialog's messages matching all the given criteria"
    ms = self.messages
    if context is None: context = 0 if headers_only else 1
    if header_section is not None:
        head = first(m for m in ms if _match_head(m, header_section))
        ms = section_msgs(ms, head) if head else Msgs()
    if context: before = after = context
    flags = re.DOTALL|re.MULTILINE|(0 if use_case else re.IGNORECASE)
    if use_regex and re_pattern:
        try: re.compile(re_pattern)
        except re.error: use_regex = False
    pat = re.compile(re_pattern if use_regex else re.escape(re_pattern), flags) if re_pattern else None
    if isinstance(ids, str): ids = [o for o in ids.split(',') if o.strip()]
    idset = {self.msg(i.strip()).id for i in ids} if ids else None
    def _txt(m): return m.content + ('\n'+m.ai_res if m.msg_type==sprompt and m.ai_res else '')
    def _ok(m):
        if headers_only and not m.header_level(): return False
        if msg_type and m.msg_type!=msg_type: return False
        if only_err and not m.has_error: return False
        if only_exp and not m.exported: return False
        if pred and not pred(m): return False
        if idset is not None and m.id not in idset: return False
        return not pat or bool(pat.search(_txt(m)))
    hits = [i for i,m in enumerate(ms) if _ok(m)]
    if limit is not None: hits = hits[:limit]
    if before or after:
        keep = set()
        for i in hits: keep.update(range(max(0,i-before), min(len(ms), i+after+1)))
        hits = sorted(keep)
    return Msgs([ms[i] for i in hits])

@delegates(Dialog.find_msgs)
def find_msgs(
    re_pattern:str='', # Regex over content (a prompt's reply included), DOTALL+MULTILINE; an invalid regex matches literally
    dlg=None, # An ipynb path; the current dialog file if None
    **kwargs,
)->MsgRows: # Snapshot rows (`id`, `msg_type`, `content`, `out`, `meta`), shown as preview lines
    "Find messages in the dialog file matching all the given criteria; for live results, call `Dialog.find_msgs`"
    return MsgRows(MsgRow(m) for m in _to_dlg(dlg).find_msgs(re_pattern, **kwargs))

`Dialog.find_msgs` returns the messages themselves, so a search on a held dialog can be edited in place; wrapping any message list in an ephemeral `Dialog` makes it searchable the same way. A prompt matches on its request or its reply; an invalid regex matches literally (search text arrives verbatim from humans); `pred=` takes any extra criterion, the channel hosts pass their own flag logic through; `header_section` follows note-message headings; `headers_only` gives an outline. `context` defaults to 1, since the neighbouring message is usually the explanation of whatever matched; pass `context=0` for exact hits only:

In [ ]:
test_eq(d.find_msgs('42', context=0), [ask])  # matches the reply text
test_eq(d.find_msgs(r'x\*2', msg_type=scode, context=0), [calc])
test_eq(d.find_msgs(only_err=True, context=0), [bad])
test_eq(len(d.find_msgs(r'x\*2')), 3)  # context defaults to 1: the match plus both neighbours
test_eq(d.find_msgs(headers_only=True), [setup])
test_eq(d.find_msgs(header_section='Setup', context=0), list(d.messages))  # everything under the heading
test_eq(Dialog(d.messages[:3]).find_msgs('42', context=0), [ask])  # an ephemeral wrap makes any message list searchable
d.find_msgs('42', context=0)[0].content = 'Double it, please.'
test_eq(ask.content, 'Double it, please.')  # live objects: the edit landed on the dialog
assert ask.dlg is d  # ...and the wrap above didn't steal the message from its real dialog
test_eq(d.find_msgs('*42*', context=0), [ask])  # an invalid regex falls back to a literal match
test_eq(d.find_msgs(pred=lambda m: m.msg_type==sprompt, context=0), [ask])  # host-specific extra criterion
xd = Dialog(name='exports')
exp1 = xd.mk_message('#| export\ndef f(): pass', msg_type=scode)
exp2 = xd.mk_message('def g(): pass', msg_type=scode)
exp2.exported = True
deep = xd.mk_message('## Deep\nbody under the heading', msg_type=snote)
test_eq(xd.find_msgs(only_exp=True, context=0), [exp1, exp2])
test_eq(xd.find_msgs(header_section='Deep', context=0), [deep])  # matched on the first line only
test_eq(xd.find_msgs(header_section='## Deep', context=0), [deep])  # a '#'-prefixed argument pins the level
test_eq(xd.find_msgs(header_section='# Deep', context=0), [])

The function form addresses a *file* and returns `MsgRow` snapshots: data to read plus the id to act on, never live objects. The file was written before the live `ask` was edited above, so the row still shows the file's content - the snapshot/live distinction in one line:

In [ ]:
rows = find_msgs('42', dlg=tdir/'demo.ipynb', context=0)
test_eq([r.id for r in rows], [ask.id])
test_eq(rows[0].content, 'Double it.')  # the file's snapshot, not the live edit above
assert not isinstance(rows[0], Message) and rows[0].msg_type==sprompt
with expect_fail(TypeError, 'path'): find_msgs('42', dlg=d)  # a live Dialog goes through the method, never dlg=
rows

174f46e0:p:Double it. ⇒ reply(13)

## Structural operations

In [ ]:
#| export
@patch
def move_msgs(self:Dialog,
    ids, # Message(s) or id(s) to move
    before=None, # Move before this message or id
    after=None, # Move after this message or id
):
    "Move messages, keeping their relative order; returns them"
    if (before is None) == (after is None): raise ValueError('Exactly one of before/after required')
    ms = Msgs([self.msg(i) for i in listify(ids)])
    self.remove_msgs(ms)
    idx = self.messages.index(self.msg(before if before is not None else after)) + (after is not None)
    self.messages[idx:idx] = ms
    return ms

def move_msgs(
    ids, # Message id(s) to move
    before=None, # Move before this message or id
    after=None, # Move after this message or id
    dlg=None, # An ipynb path; the current dialog file if None
):
    "Move messages in the dialog file, keeping their relative order; returns them"
    d = _to_dlg(dlg)
    res = d.move_msgs(ids, before=before, after=after)
    _save(d)
    return res

In [ ]:
#| export
def _atts_for(content, att_map):
    "The attachments from `att_map` whose ids are referenced in `content`"
    return [a for aid,a in att_map.items() if aid in content]

@patch
def split(self:Message,
    *linenos:int, # Split before each of these 1-based lines
):
    "Split this message into pieces (returned as `Msgs`; the first keeps its id): one `'\\n\\n'` is absorbed at each cut (so `merge_msgs` restores blank-line-separated content byte-exactly), `meta_attrs` fields, the cell `meta`, and a leading `#| export` copy to every piece, attachments follow their references, and unreferenced ones stay on the first piece"
    d = self.dlg
    if d is None: raise ValueError('message is not in a dialog')
    lines = self.content.splitlines()
    cuts = [0, *[l-1 for l in linenos], len(lines)]
    parts = ['\n'.join(lines[a:b]) for a,b in zip(cuts, cuts[1:])]
    for i in range(len(parts)-1):  # each cut absorbs the '\n\n' that a later merge re-inserts
        if   parts[i].endswith('\n'):     parts[i] = parts[i][:-1]
        elif parts[i+1].startswith('\n'): parts[i+1] = parts[i+1][1:]
    src = self.content
    keep = {a: getattr(self, a) for a in self.meta_attrs if hasattr(self, a)}
    att_map = {a.id: a for a in self.attachments}
    refs = [_atts_for(p, att_map) for p in parts]
    used = {a.id for r in refs for a in r}
    refs[0] += [a for a in self.attachments if a.id not in used]
    self.content, self.attachments = parts[0], refs[0]
    prev, res = self, [self]
    for p,r in zip(parts[1:], refs[1:]):
        p = copy_export(p, src)
        prev = d.mk_message(p, after=prev, msg_type=self.msg_type, attachments=r, meta=copy.deepcopy(self.meta), **keep)
        res.append(prev)
    return Msgs(res)

def split_msg(
    id, # Message id to split
    *linenos:int, # Split before each of these 1-based lines
    dlg=None, # An ipynb path; the current dialog file if None
):
    "Split a message in the dialog file (see `Message.split`)"
    d = _to_dlg(dlg)
    res = d.msg(id).split(*linenos)
    _save(d)
    return res

@patch
def merge_msgs(self:Dialog,
    *ids, # Adjacent messages or ids to merge
):
    "Merge into the first message (returned, keeping its id): same-type merges keep the type, mixed become a note via `merge_content`; metas and directives merge first-wins via `merge_parts`, outputs clear, attachments combine"
    ms = [self.msg(i) for i in ids]
    mtype = ms[0].msg_type if len(set(m.msg_type for m in ms))==1 else snote
    content, meta = merge_parts(ms, [m.merge_content(mtype) for m in ms])
    ms[0].update(content=content, meta=meta, output=None, msg_type=mtype, attachments=L(ms).attrgot('attachments').concat())
    self.remove_msgs(ms[1:])
    return ms[0]

def merge_msgs(
    *ids, # Adjacent message ids to merge
    dlg=None, # An ipynb path; the current dialog file if None
):
    "Merge messages in the dialog file (see `Dialog.merge_msgs`)"
    d = _to_dlg(dlg)
    res = d.merge_msgs(*ids)
    _save(d)
    return res

In [ ]:
sd = Dialog(name='s')
sm = sd.mk_message('#| export\ndef f(): pass\ndef g(): pass', msg_type=scode, meta={'k': 1})
i1,i2 = sm.split(3)
test_eq([m.content for m in sd.messages], ['#| export\ndef f(): pass', '#| export\ndef g(): pass'])
test_eq([m.meta for m in sd.messages], [{'k': 1}, {'k': 1}])  # whole cell meta copies to every piece
i2.meta['k'] = 2  # pieces don't share the dict
test_eq(i1.meta, {'k': 1})
sd.merge_msgs(i1, i2)
test_eq((sd.messages[0].content, sd.messages[0].meta), ('#| export\ndef f(): pass\n\ndef g(): pass', {'k': 1}))
hd = Dialog(name='hoist')
h1, h2 = hd.mk_message('def a(): pass'), hd.mk_message('#| export\ndef b(): pass')
hm = hd.merge_msgs(h1, h2)  # any exported piece exports the merge, via meta
test_eq((hm.content, hm.meta, hm.exported), ('def a(): pass\n\ndef b(): pass', {'nbdev': {'export': 'true'}}, True))
sd.mk_message('tail', msg_type=snote)
sd.move_msgs(i1, after=sd.messages[-1])
test_eq([m.content[:4] for m in sd.messages], ['tail', '#| e'])

In [ ]:
ad = Dialog(name='atts')
at1, at2 = Attachment(b'x', 'text/plain'), Attachment(b'y', 'text/plain')
am = ad.mk_message(f'see attachment:{at1.id}\nplain tail', msg_type=snote, attachments=[at1, at2])
p1, p2 = am.split(2)
test_eq(len(p1.attachments), 2)  # the referenced one, plus the unreferenced one kept on the first piece
test_eq(p2.attachments, [])
jd = Dialog(name='junction')
j = jd.mk_message('a\n\n\nb', msg_type=snote)
j1, j2 = j.split(4)
test_eq((j1.content, j2.content), ('a\n', 'b'))  # one '\n\n' absorbed at the cut: exactly what merge re-inserts
test_eq(jd.merge_msgs(j1, j2).content, 'a\n\n\nb')  # ...so blank-line-separated content round trips byte-exactly
k1, k2 = jd.mk_message('a\n\nb', msg_type=snote).split(3)
test_eq((k1.content, k2.content), ('a', 'b'))
w1, w2 = jd.mk_message('a\n\n\nb', msg_type=snote).split(3)  # cut on the blank line: residue stays on the right
test_eq((w1.content, w2.content), ('a', '\nb'))
with expect_fail(ValueError, 'Exactly one'): jd.move_msgs(w2)

In [ ]:
#| export
_paste_buf = []

@patch
def copy_msgs(self:Dialog,
    *ids, # Messages or ids to copy
):
    "Copy messages into the paste buffer (replacing its contents), for later `paste_msgs`"
    global _paste_buf
    _paste_buf = Msgs([self.msg(i) for i in ids])
    return _paste_buf

@patch
def cut_msgs(self:Dialog,
    *ids, # Messages or ids to cut
):
    "Copy messages into the paste buffer, then remove them from the dialog"
    res = self.copy_msgs(*ids)
    self.remove_msgs(res)
    return res

@patch
def paste_msgs(self:Dialog,
    before=None, # Insert before this message or id
    after=None, # Insert after this message or id
):
    "Insert copies of the buffered messages (fresh ids) before/after a message or id; returns the new messages"
    if before is not None: before = self.msg(before)
    if after is not None: after = self.msg(after)
    return Msgs(self.mk_messages(_paste_buf, before=before, after=after))

def copy_msgs(
    *ids, # Message ids to copy
    dlg=None, # An ipynb path; the current dialog file if None
):
    "Copy messages into the paste buffer (replacing its contents), for later `paste_msgs`"
    return _to_dlg(dlg).copy_msgs(*ids)

def cut_msgs(
    *ids, # Message ids to cut
    dlg=None, # An ipynb path; the current dialog file if None
):
    "Copy messages into the paste buffer, then remove them from the dialog file"
    d = _to_dlg(dlg)
    res = d.cut_msgs(*ids)
    _save(d)
    return res

def paste_msgs(
    before=None, # Insert before this message or id
    after=None, # Insert after this message or id
    dlg=None, # An ipynb path; the current dialog file if None
):
    "Insert copies of the buffered messages (fresh ids) before/after a message or id in the dialog file; returns the new messages"
    d = _to_dlg(dlg)
    res = d.paste_msgs(before=before, after=after)
    _save(d)
    return res

In [ ]:
sd.copy_msgs(i1)
pasted = sd.paste_msgs(after=i1)
test_eq(len(sd.messages), 3)
assert pasted[0].id != sd.msg(i1).id and pasted[0].content == sd.msg(i1).content
sd.cut_msgs(pasted[0].id)
test_eq(len(sd.messages), 2)

## Editing message text

The editing operations run fastcore.tools' text primitives over a message's content, or, with `out=True`, over a prompt's reply markdown (assignment re-wraps it through `prompt_output`). Only prompts have text-editable output: for a code message's outputs, assign `msg.output` directly. The old behavior of editing a prompt cell's joined request+reply source is retired. In bulk mode (a list of ids, or 'all'), no-ops are dropped from the result but errors are reported.

In [ ]:
#| export
def _out_text(m):
    "A prompt's reply markdown, the only output editable as text"
    if m.msg_type != sprompt: raise ValueError(f"{m.id}: out=True edits a prompt's reply; assign `msg.output` directly for other types")
    return m.ai_res

def _set_out_text(m, s): m.output = s

In [ ]:
#| export
_msg_edit_doc = """
Be sure you've viewed the message (e.g. `view_msg`) so you know the line nums.

Message editing standard parameters:
- `id`: message id (or list of ids, or 'all'), unique prefixes allowed
- `dlg`: an ipynb path; the current dialog file if None
- `out`: edit the output text (prompt reply, or code outputs literal) instead of the source

returns: diff of changes, or "none: No changes.", or "error: ..."
"""

def _msg_edit(f, name=None):
    def wrapper(id, *args, dlg=None, out:bool=False, **kw):
        d, sv = _load_dlg(dlg)
        def _one(cid):
            m = d.msg(cid)
            try:
                text = _out_text(m) if out else m.content
                if not text: return f"error: Message has no {'output' if out else 'source'}"
                new = f(text, *args, **kw)
                if out: _set_out_text(m, new)
                else: m.content = new
            except Exception as e: return f'error: {e}'
            return str_diff(text, new) or 'none: No changes.'
        if isinstance(id, list) or id=='all':
            if id=='all': id = [m.id for m in d.messages]
            res = [(cid, r) for cid in id if not (r := _one(cid)).startswith('none:')]
        else: res = PrettyString(_one(id))
        _save(d, sv)
        return res
    res = splice_sig(wrapper, f, 'text')
    if name: res.__name__ = res.__qualname__ = name
    res.__doc__ = (f.__doc__ or '') + _msg_edit_doc
    return res

msg_insert_line   = _msg_edit(insert_line,   'msg_insert_line')
msg_str_replace   = _msg_edit(str_replace,   'msg_str_replace')
msg_strs_replace  = _msg_edit(strs_replace,  'msg_strs_replace')
msg_replace_lines = _msg_edit(replace_lines, 'msg_replace_lines')
msg_del_lines     = _msg_edit(del_lines,     'msg_del_lines')
msg_ast_replace   = _msg_edit(ast_replace,   'msg_ast_replace')

def _msg_method(f):
    def method(self, *args, out:bool=False, **kw):
        text = _out_text(self) if out else self.content
        if not text: return PrettyString(f"error: Message has no {'output' if out else 'source'}")
        try: new = f(text, *args, **kw)
        except ValueError as e: return PrettyString(f'error: {e}')
        if out: _set_out_text(self, new)
        else: self.content = new
        return PrettyString(str_diff(text, new) or 'none: No changes.')
    res = splice_sig(method, f, 'text')
    res.__name__ = res.__qualname__ = f.__name__
    res.__doc__ = (f.__doc__ or '') + "\nIn-memory edit of this message (`out=True` edits a prompt's reply), returning a diff; nothing is saved."
    return res

for _f in (insert_line, str_replace, strs_replace, replace_lines, del_lines, ast_replace): setattr(Message, _f.__name__, _msg_method(_f))

The spliced editors inherit `fastcore.tools`' signatures, so the toolkit-wide `replace_params` group (declared identically in `fastcore.editskill`) collapses the same six options in `doc()` overviews here:

In [ ]:
#| export
__pyskill_params__ = {'replace_params': ('start_line', 'end_line', 'n_matches', 're_filter', 'invert_filter', 'use_regex')}

In [ ]:
(tdir/'edit.ipynb').write_bytes((tdir/'demo.ipynb').read_bytes())  # scratch copy: demo.ipynb stays pristine for later sections
set_dlg(tdir/'edit.ipynb')
diff = msg_str_replace(calc.id, 'x = 21', 'x = 40')
assert '-x = 21' in str(diff) and '+x = 40' in str(diff)
test_eq(read_ipynb(tdir/'edit.ipynb').msg(calc.id).content, 'x = 40\nx*2')

In [ ]:
diff = msg_str_replace(ask.id, '**42**', '**80**', out=True)
assert '+It is **80**.' in str(diff)
test_eq(read_ipynb(tdir/'edit.ipynb').msg(ask.id).ai_res, 'It is **80**.')
assert str(msg_str_replace(calc.id, "'42'", "'80'", out=True)).startswith('error:')
assert str(msg_del_lines(setup.id, 1, 1, out=True)).startswith('error:')
bulk = msg_str_replace([setup.id, bad.id], '1/0', 'pass')
test_eq(len(bulk), 2)  # bulk mode reports errors alongside diffs
assert 'error' in bulk[0][1] and '+pass' in bulk[1][1]

## Structural search

Text search fights Python syntax: `cfg =` misses `cfg, aux = load()`, and a loose pattern on a two-letter name floods. The finder trio turns remold's structural queries into predicates for `find_msgs(pred=)`: `symdef_finder` matches messages that bind a name in their top-level scope, however Python spells the binding (tuple unpacking, `for`, `with ... as`, walrus, imports, `def`/`class`); `symref_finder` matches those that read it; and `ast_finder` matches an ast-grep pattern. A message that doesn't parse as Python never matches, so prose cells exclude themselves.

In [ ]:
#| export
def symdef_finder(name):
    "`find_msgs` predicate: does the message bind `name` at cell top level? (requires `remold`)"
    from remold import symdefs
    return lambda m: name in symdefs(m.content)

def symref_finder(name):
    "`find_msgs` predicate: does the message reference `name`? (requires `remold`)"
    from remold import symrefs
    return lambda m: name in symrefs(m.content)

def ast_finder(pattern):
    "`find_msgs` predicate: does ast-grep `pattern` match in the message? (requires `remold`)"
    from remold import astfind
    return lambda m: bool(astfind(m.content, pattern))

In [ ]:
fd = Dialog(name='finders')
f1 = fd.mk_message('cfg, aux = load_cfg()', msg_type=scode)
f2 = fd.mk_message('print(render(cfg))', msg_type=scode)
fd.mk_message('cfg is set up above', msg_type=snote)
test_eq(fd.find_msgs(pred=symdef_finder('cfg'), context=0), [f1])
test_eq(fd.find_msgs(pred=symref_finder('cfg'), context=0), [f2])
test_eq(fd.find_msgs(pred=ast_finder('render($_)'), context=0), [f2])

## Hash-verified edits

`m.lnhashview()` and `m.exhash()` bring exhash's stale-context-proof addressing to a held message; `lnhashview_msg`/`msg_exhash` are their file-transaction twins, taking an id and `dlg=` path. `Message.lnhashview` is sugar for `view(lnhashs=True)`; the exhash pair imports `exhash` lazily, so llmsurgery carries no hard dependency on it:

In [ ]:
#| export
@patch
def lnhashview(self:Message):
    "Hash-addressed view of this message's content, for `Message.exhash`"
    return self.view(lnhashs=True)

@patch
def exhash(self:Message,
    *cmds:tuple, # exhash command tuples, addresses from `lnhashview`
    sw:int=4, # Shift width for indent commands
):
    "Apply exhash commands to this message's content in memory, returning the diff"
    from exhash import exhash as _exhash
    res = _exhash(self.content, list(cmds), sw=sw)
    self.content = '\n'.join(res.lines)
    return PrettyString(res.format_diff(context=1))

def lnhashview_msg(
    id, # Message id, looked up in `dlg` (unique prefixes allowed)
    dlg=None, # An ipynb path; the current dialog file if None
):
    "Hash-addressed view of a message's content, for `msg_exhash`"
    return _to_dlg(dlg).msg(id).lnhashview()

def msg_exhash(
    id, # Message id, looked up in `dlg` (unique prefixes allowed)
    *cmds:tuple, # exhash command tuples, addresses from `lnhashview_msg`
    sw:int=4, # Shift width for indent commands
    dlg=None, # An ipynb path; the current dialog file if None
):
    "Apply exhash commands to a message in the dialog file, returning the diff"
    d = _to_dlg(dlg)
    res = d.msg(id).exhash(*cmds, sw=sw)
    _save(d)
    return res

In [ ]:
diff = calc.exhash((lnhash(1, 'x = 21'), 's', '21', '12'))
test_eq(calc.content, 'x = 12\nx*2')
with expect_fail(Exception, 'stale'): calc.exhash((lnhash(1, 'wrong text'), 'd'))

The transactional twins take a message id, resolved through `dlg` like `view_msg` (the current dialog file when None), and save back to the file automatically:

In [ ]:
msg_replace_lines(calc.id, new_content='a = 1\na*2')
assert 'a = 1' in str(lnhashview_msg(calc.id))
diff = msg_exhash(calc.id, (lnhash(1, 'a = 1'), 's', '1', '9'))
test_eq(read_ipynb(tdir/'edit.ipynb').msg(calc.id).content, 'a = 9\na*2')

## Working with dialog files

Every function above takes `dlg` as an ipynb path, or None meaning the current dialog file (`set_dlg`); each call reads the file fresh, applies, and saves - a transaction. A held `Dialog` never goes through `dlg=`: its methods are the session surface, mutations stay in memory, and `dlg.save()` commits (`cur_dlg()` opens a fresh session on the current file). The two shapes don't see each other mid-flight, so use one at a time per file: save before switching to functions, reopen after. `write_ipynb`'s byte-stability keeps the resulting diffs minimal. A few functions below exist only for the file world:

In [ ]:
set_dlg(tdir/'demo.ipynb')
assert '<prompt' in str(view_dlg())
assert str(summary_dlg()).count('\n')==3
s = str(find_msgs(r'x\*2', msg_type=scode, context=0))
assert s and s.count('\n')==0  # rows display as preview lines: exactly one (empty would pass a bare count check)
xd.mk_message('', msg_type=sraw, meta=dict(rec_kind='system', rec={}))
assert 'kind="system"' in str(xd.view())  # tagged raws are badged, never shown as ordinary content

In [ ]:
with working_directory(tdir):
    set_dlg('demo.ipynb')
    with working_directory('/'): assert cur_dlg() is not None  # registration survives later cwd changes

In [ ]:
#| export
def add_msg(
    source:str, # source for the new message
    msg_type:str='code', # 'code', 'note', 'prompt', or 'raw'
    before:str=None, # message or id to insert before
    after:str=None, # message or id to insert after
    dlg=None, # An ipynb path; the current dialog file if None
):
    "Add a new message before/after an existing one (pass exactly one), returning it"
    d, sv = _load_dlg(dlg)
    m = d.mk_message(source, msg_type=msg_type,
        before=None if before is None else d.msg(before), after=None if after is None else d.msg(after))
    _save(d, sv)
    return m

def del_msgs(
    *ids, # Messages or ids to delete
    dlg=None, # An ipynb path; the current dialog file if None
):
    "Delete messages by id, returning the removed messages"
    d, sv = _load_dlg(dlg)
    res = Msgs(d.remove_msgs([d.msg(i) for i in ids]))
    _save(d, sv)
    return res

def create_dlg(
    fname:str, # path for the new ipynb; must not already exist
    source:str='', # source for its first message
    msg_type:str='code', # 'code', 'note', 'prompt', or 'raw'
):
    "Create a new dialog file containing one message, returning the `Dialog` (with `path_` stamped)"
    f = Path(fname)
    if f.exists(): raise FileExistsError(str(f))
    d = Dialog(name=f.stem)
    d.mk_message(source, msg_type=msg_type)
    write_ipynb(d, f)
    d.path_ = f
    return d

In [ ]:
nm = add_msg('print(9)', after=calc.id)
test_eq(read_ipynb(tdir/'demo.ipynb').msg(nm).content, 'print(9)')
test_eq(del_msgs(nm)[0].content, 'print(9)')
with expect_fail(KeyError, 'no message'): read_ipynb(tdir/'demo.ipynb').msg(nm)
nd = create_dlg(tdir/'new.ipynb', '# hi', 'note')
test_eq(nd.messages[0].msg_type, snote)
test_eq(read_ipynb(tdir/'new.ipynb').msg(nd.messages[0]).content, '# hi')

In [ ]:
s1, s2 = split_msg(calc.id, 2)
test_eq(read_ipynb(tdir/'demo.ipynb').msg(s2).content, 'x*2')
merge_msgs(s1, s2)
test_eq(read_ipynb(tdir/'demo.ipynb').msg(s1).content, 'x = 21\n\nx*2')
copy_msgs(s1)
pids = paste_msgs(after=s1)
test_eq(read_ipynb(tdir/'demo.ipynb').msg(pids[0].id).content, 'x = 21\n\nx*2')
del_msgs(pids[0].id)

9b80fe7c:c:x = 21\n\nx*2

Under IPython, the `%%add_msg` magic takes the message body verbatim (no Python quoting): `%%add_msg [<path>] before=<id>|after=<id> [code|note|prompt|raw]`. Keyword spellings (`dlg=`, `msg_type=`) are accepted too, and always win over bare tokens:

In [ ]:
#| export
def add_msg_magic(line, cell):
    "Add a new message with the magic body as its source, taken verbatim."
    kw = {}
    for t in shlex.split(line):
        if '=' in t: kw.update([t.split('=', 1)])
        elif t in smsg_types: kw.setdefault('msg_type', t)
        else: kw.setdefault('dlg', t)
    return add_msg(cell.rstrip('\n'), **kw)

def load_ipython_extension(ipython): ipython.register_magic_function(add_msg_magic, 'cell', 'add_msg')

try:
    from IPython import get_ipython
    if (_ip := get_ipython()): load_ipython_extension(_ip)
except ImportError: pass

In [ ]:
nm = add_msg_magic(f"note dlg={tdir/'demo.ipynb'} after={calc.id}", 'magic body')
test_eq(read_ipynb(tdir/'demo.ipynb').msg(nm.id).content, 'magic body')
test_eq(nm.msg_type, 'note')